# 021 — Representación del conocimiento y ontologías

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

La solución valida el contrato mínimo sin asumir un valor interno específico.


In [ ]:
result = run_lab("logic", seed=21)
assert result["kind"] == "logic"
assert result["evidence"]
show(result)


## Solución 1 — Relaciones

a) **instance-of**: GPT-4 es un individuo de la clase ModeloDeLenguaje; hereda
las propiedades de la clase.
b) **is-a**: subclase entre clases; ModeloDeLenguaje hereda todo lo declarado
para ModeloML.
c) **part-of**: composición; la capa NO hereda propiedades del Transformer
(no "procesa lenguaje de punta a punta" por ser parte de algo que sí).
d) **instance-of**: Rex es individuo de PastorAleman (que a su vez `is-a` Perro).
La prueba rápida: ¿tiene sentido decir "Rex tiene 4 patas" por herencia? Sí →
instancia. ¿"La capa de atención tiene 175B de parámetros" por ser parte? No →
part-of no hereda.


## Solución 2 — Excepciones

a) Herencia ingenua desde Pingu: `Pinguino` ya declara `vuela=false`, pero un
algoritmo que suba hasta el primer nivel que conozca (mal implementado, desde
`Ave`) diría `true`. Con "el más específico gana": Pinguino (distancia 1)
declara `false` antes que Ave (distancia 2) → **false**, correcto.

b) Cualquier valor es defendible: `false` (sigue siendo pingüino) parece obvio,
pero el punto es que **el algoritmo no puede decidirlo**: es una decisión de
modelado del dominio. Las cadenas de excepciones anidadas son la razón por la
que los sistemas de herencia por defecto exigen criterios explícitos de
precedencia (y por la que OWL, monótono, directamente no permite excepciones).


In [ ]:
triples = [
    ("Pinguino", "subclase_de", "Ave"),
    ("Ave", "subclase_de", "Animal"),
    ("Ave", "vuela", True),
    ("Pinguino", "vuela", False),
    ("Pingu", "instancia_de", "Pinguino"),
]
def vuela(entidad):
    nivel = next((o for s, p, o in triples if s == entidad and p == "instancia_de"), entidad)
    while nivel is not None:
        declarado = [o for s, p, o in triples if s == nivel and p == "vuela"]
        if declarado:
            return declarado[0]
        nivel = next((o for s, p, o in triples if s == nivel and p == "subclase_de"), None)
    return None

assert vuela("Pingu") is False
print("el nivel más específico (Pinguino) gana: Pingu no vuela ✔")


## Solución 3 — Tripletas

```text
Datos (individuos):
(curso021, pertenece_a, parte1)
(parte1,  parte_de,     programaIA)
(parte1,  proyecto_final, curso024)

Esquema (clases):
(Curso, es_clase, true)            (Parte, es_clase, true)
(curso021, instancia_de, Curso)    (parte1, instancia_de, Parte)
(Parte, tiene_propiedad_obligatoria, proyecto_final)   ← axioma de cardinalidad
```

La frase "toda parte tiene un proyecto final" es **esquema** (axioma sobre la
clase Parte, en OWL una restricción de cardinalidad); "el proyecto de la parte
1 es la clase 024" es **dato**. Confundir los dos niveles es el error de
modelado más común en grafos de conocimiento.


## Solución 4 — Reglas como axiomas

`facts` se leería como tripletas del individuo proyecto:
`(proyecto, tiene_datos, true)`, `(proyecto, puede_experimentar, true)`, etc.
Las reglas del motor juegan el papel de los axiomas OWL: así como una
propiedad transitiva hace que el razonador derive tripletas nuevas
(`part_of` encadenado), cada regla `if→then` deriva hechos que nadie escribió
explícitamente. La diferencia honesta: OWL garantiza decidibilidad y
consistencia global verificable; un motor de reglas ad hoc solo garantiza el
punto fijo de SUS reglas.


In [ ]:
result = run_lab("logic", seed=21)
derivados = [f["then"] for f in result["result"]["rules_fired"]]
print("tripletas derivadas por los 'axiomas':", derivados)


## Reflexión

1. En la micro-ontología, ¿por qué 'la excepción más específica gana' rompe la monotonía de la lógica clásica y qué problema práctico aparece cuando dos excepciones a distinta altura entran en conflicto?
2. Wikidata usa tripletas; MYCIN usaba reglas; STRIPS usa literales. ¿Qué decide qué representación conviene: el dominio, la consulta o el mecanismo de inferencia disponible?
3. ¿Qué error de modelado comete quien declara (Rueda, subclase_de, Coche) y qué inferencias absurdas produciría? ¿Cuál es la relación correcta?
